In [70]:
import numpy as np
import pandas as pd
import math
import random
import matplotlib.pyplot as plt

# 1. Definição do dataset de nós (Hospital e Pontos de Entrega)
dados_locais = {
    'Local': ['Hospital', 'Unidade A', 'Casa B', 'Unidade C', 'Casa D', 'Unidade E'],
    'X': [0, 2, 5, 1, 8, 3],
    'Y': [0, 4, 2, 7, 5, 1],
    'Demanda_Caixas': [0, 10, 5, 15, 8, 12],
    'Prioridade': ['Origem', 'Alta', 'Baixa', 'Alta', 'Média', 'Baixa']
}

df_locais = pd.DataFrame(dados_locais)

# 2. Definição das restrições do Problema de Roteamento de Veículos (VRP)
CAPACIDADE_MAXIMA_VEICULO = 20  # Restrição de capacidade por veículo
AUTONOMIA_MAXIMA = 25           # Restrição de distância máxima (km) por rota

print("Bloco 1 executado com sucesso! Dados e restrições carregados.")

Bloco 1 executado com sucesso! Dados e restrições carregados.


## 🎯 1. O Desafio e as Regras do Jogo (Modelagem)
Olá! Neste primeiro bloco, nós desenhamos o cenário do nosso problema logístico. É como se estivéssemos montando o tabuleiro do jogo.

Simulamos o nosso hospital e os pontos de entrega na cidade de São José dos Campos. Para que o projeto não ficasse apenas na teoria, aplicamos **restrições reais do dia a dia (Hard Constraints)**, ou seja, regras inquebráveis para a nossa frota:
* **📦 Limite de Porta-Malas (Capacidade):** Cada veículo só leva no máximo 20 caixas.
* **⛽ Limite de Combustível (Autonomia):** Nenhum motorista pode rodar mais que 25 km por rota.

In [71]:
# Pré-processamento para otimização de consultas durante a execução
coords = list(zip(df_locais['X'], df_locais['Y']))
demandas = df_locais['Demanda_Caixas'].tolist()
prioridades = df_locais['Prioridade'].tolist()

def calcular_distancia_euclidiana(p1, p2):
    """Calcula a distância em linha reta entre duas coordenadas cartesianas."""
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def decodificar_rota_vrp(cromossomo):
    """
    Decodifica o gene (permutação de nós) em rotas válidas para os veículos,
    respeitando estritamente a restrição de capacidade máxima.
    """
    viagens = []
    viagem_atual = []
    carga_atual = 0

    for local in cromossomo:
        demanda_local = demandas[local]
        if carga_atual + demanda_local > CAPACIDADE_MAXIMA_VEICULO:
            viagens.append(viagem_atual)
            viagem_atual = [local]
            carga_atual = demanda_local
        else:
            viagem_atual.append(local)
            carga_atual += demanda_local

    if viagem_atual:
        viagens.append(viagem_atual)
    return viagens

def calcular_fitness_vrp(cromossomo):
    """
    Avalia a qualidade da rota.
    Minimiza a distância total, aplica pesos para prioridades (Alta = maior peso logístico)
    e penaliza severamente rotas que violam a autonomia máxima de 25km.
    """
    viagens = decodificar_rota_vrp(cromossomo)
    distancia_total = 0
    penalidades = 0

    for viagem in viagens:
        dist_viagem = 0
        dist_acumulada = 0
        local_anterior = 0 # Início sempre no nó 0 (Hospital)

        for local_atual in viagem:
            dist = calcular_distancia_euclidiana(coords[local_anterior], coords[local_atual])
            dist_viagem += dist
            dist_acumulada += dist

            # Penalização baseada em SLA de atendimento (Prioridade)
            peso_prioridade = 2.0 if prioridades[local_atual] == 'Alta' else 0.5
            penalidades += (dist_acumulada * peso_prioridade)

            local_anterior = local_atual

        # Retorno ao Hospital
        dist_viagem += calcular_distancia_euclidiana(coords[local_anterior], coords[0])
        distancia_total += dist_viagem

        # Penalidade severa (Hard Constraint) por violação de autonomia
        if dist_viagem > AUTONOMIA_MAXIMA:
            penalidades += 10000 * (dist_viagem - AUTONOMIA_MAXIMA)

    custo_total = distancia_total + penalidades
    return 1.0 / (custo_total + 1e-6) # Retorna o inverso do custo (maximiza o fitness)

print("Bloco 2 executado com sucesso! Funções matemáticas e de avaliação (Fitness) criadas.")

Bloco 2 executado com sucesso! Funções matemáticas e de avaliação (Fitness) criadas.


## 🧠 2. O Sistema de Punição e Recompensa (Função Fitness)
Como o computador sabe qual rota é a melhor? Neste bloco, criamos a **Função Fitness**, que funciona como um sistema de pontuação ou "multa invisível".

Em vez de apenas procurar o caminho mais curto, o algoritmo age como um gerente financeiro:
1. Ele **dá notas baixas (penaliza)** se o trajeto atrasar a entrega de insumos de alta prioridade.
2. Ele **aplica uma punição severa** se a rota estourar o limite de 25km.
Isso garante que o sistema não crie soluções que seriam impossíveis ou caras demais na vida real.

In [72]:
# Hiperparâmetros do Algoritmo Genético
TAMANHO_POPULACAO = 50
GERACOES = 100
TAXA_MUTACAO = 0.1

def inicializar_populacao(tamanho):
    """Gera população inicial com permutações aleatórias dos pontos de entrega (1 a 5)."""
    locais_entrega = [1, 2, 3, 4, 5]
    return [random.sample(locais_entrega, len(locais_entrega)) for _ in range(tamanho)]

def selecao_torneio(populacao, fitness_pop, k=3):
    """Seleciona o indivíduo mais apto em um subgrupo (torneio)."""
    selecionados = random.sample(list(zip(populacao, fitness_pop)), k)
    return max(selecionados, key=lambda x: x[1])[0]

def crossover_ordem_ox1(pai1, pai2):
    """
    Order Crossover (OX1): Ideal para problemas de roteamento pois
    preserva a ordem e evita nós duplicados ou faltantes.
    """
    tamanho = len(pai1)
    inicio, fim = sorted(random.sample(range(tamanho), 2))

    filho = [None] * tamanho
    filho[inicio:fim] = pai1[inicio:fim]

    p2_filtrado = [x for x in pai2 if x not in filho]
    idx_p2 = 0
    for i in range(tamanho):
        if filho[i] is None:
            filho[i] = p2_filtrado[idx_p2]
            idx_p2 += 1
    return filho

def mutacao_swap(individuo):
    """Troca a posição de dois nós aleatoriamente para manter a diversidade genética."""
    if random.random() < TAXA_MUTACAO:
        i, j = random.sample(range(len(individuo)), 2)
        individuo[i], individuo[j] = individuo[j], individuo[i]
    return individuo

# ================= EXECUÇÃO DO ALGORITMO =================
populacao = inicializar_populacao(TAMANHO_POPULACAO)
melhor_solucao_global = None
melhor_fitness_global = -1

for geracao in range(GERACOES):
    fitness_pop = [calcular_fitness_vrp(ind) for ind in populacao]

    for i, ind in enumerate(populacao):
        if fitness_pop[i] > melhor_fitness_global:
            melhor_fitness_global = fitness_pop[i]
            melhor_solucao_global = ind

    nova_populacao = []

    while len(nova_populacao) < TAMANHO_POPULACAO:
        pai1 = selecao_torneio(populacao, fitness_pop)
        pai2 = selecao_torneio(populacao, fitness_pop)
        filho = mutacao_swap(crossover_ordem_ox1(pai1, pai2))
        nova_populacao.append(filho)

    populacao = nova_populacao

melhores_viagens_vrp = decodificar_rota_vrp(melhor_solucao_global)

print("Bloco 3 executado com sucesso! Evolução concluída.")
print(f"Melhor rota encontrada (Cromossomo): {melhor_solucao_global}")
print(f"Rotas divididas por veículos: {melhores_viagens_vrp}")

Bloco 3 executado com sucesso! Evolução concluída.
Melhor rota encontrada (Cromossomo): [3, 1, 4, 5, 2]
Rotas divididas por veículos: [[3], [1, 4], [5, 2]]


## ⚙️ 3. A Seleção Natural (Algoritmo Genético)
Aqui está o coração da Inteligência Artificial do projeto. Para achar a rota perfeita, nós imitamos a evolução da natureza usando um **Algoritmo Genético**.

Funciona assim, ao longo de 100 gerações:
* **🏆 Torneio:** O sistema escolhe as rotas que gastam menos tempo e dinheiro.
* **🧬 Cruzamento (Crossover OX1):** Pegamos a "metade boa" do caminho do Veículo A e misturamos com a "metade boa" do Veículo B, sem esquecer nenhum endereço.
* **🔄 Mutação:** De vez em quando, o sistema troca dois endereços de ordem de propósito (fator surpresa), para ver se descobre um atalho que não tínhamos pensado.

In [73]:
# ================= VISUALIZAÇÃO DOS RESULTADOS (MAPA REAL) =================
# Instala o folium caso não esteja no ambiente
!pip install folium

import folium

# 1. Novas Coordenadas (Latitude e Longitude reais em São José dos Campos)
# Hospital Universitário (Fictício, perto do centro)
# Unidades e Casas espalhadas pela cidade (ex: Aquarius, Satélite, etc)
dados_locais_reais = {
    'Local': ['Hospital Central', 'Unidade A (Aquarius)', 'Casa B (Satélite)', 'Unidade C (Centro)', 'Casa D (Urbanova)', 'Unidade E (Vista Verde)'],
    'Lat': [-23.2198, -23.2250, -23.2350, -23.1850, -23.2000, -23.2050],
    'Lon': [-45.8900, -45.9000, -45.8850, -45.8800, -45.9150, -45.8650],
    'Demanda_Caixas': [0, 10, 5, 15, 8, 12],
    'Prioridade': ['Origem', 'Alta', 'Baixa', 'Alta', 'Média', 'Baixa']
}

# 2. Criando o mapa centralizado em SJC
mapa_sjc = folium.Map(
    location=[-23.2100, -45.8900],
    zoom_start=13,
    tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
    attr='Google Maps'
)
cores_rotas = ['red', 'blue', 'green']

# 3. Adicionando marcadores das unidades e casas
for i in range(len(dados_locais_reais['Local'])):
    icone_cor = 'black' if i == 0 else 'lightgray'
    icone_tipo = 'plus' if i == 0 else 'info-sign'

    folium.Marker(
        [dados_locais_reais['Lat'][i], dados_locais_reais['Lon'][i]],
        popup=f"<b>{dados_locais_reais['Local'][i]}</b><br>Demanda: {dados_locais_reais['Demanda_Caixas'][i]} caixas<br>Prioridade: {dados_locais_reais['Prioridade'][i]}",
        icon=folium.Icon(color=icone_cor, icon=icone_tipo)
    ).add_to(mapa_sjc)

# 4. Traçando as rotas dos veículos
for i, viagem in enumerate(melhores_viagens_vrp):
    # O veículo sai do Hospital
    rota_coordenadas = [[dados_locais_reais['Lat'][0], dados_locais_reais['Lon'][0]]]

    # Passa pelos pontos da viagem
    for loc_idx in viagem:
        rota_coordenadas.append([dados_locais_reais['Lat'][loc_idx], dados_locais_reais['Lon'][loc_idx]])

    # E retorna ao Hospital
    rota_coordenadas.append([dados_locais_reais['Lat'][0], dados_locais_reais['Lon'][0]])

    # Desenha a linha no mapa
    folium.PolyLine(
        rota_coordenadas,
        color=cores_rotas[i],
        weight=4,
        opacity=0.8,
        tooltip=f"Rota do Veículo {i+1} (Carga: {sum([demandas[x] for x in viagem])} cx)"
    ).add_to(mapa_sjc)

mapa_sjc.save('mapa_final.html')

# Exibe o mapa dentro do Colab
mapa_sjc



## 🗺️ 4. O Mapa na Tela (Visualização no Mundo Real)
Para provar que toda essa matemática funciona, nós tiramos os números do papel e jogamos no mapa real usando a biblioteca **Folium**.

Ao olhar para o mapa com as ruas reais, conseguimos ver que o algoritmo é inteligente. Ele criou um agrupamento perfeito (**clustering**).
Em vez de fazer os motoristas ziguezaguearem pela cidade cruzando o caminho uns dos outros, ele dividiu as entregas por bairros (cada cor é um veículo). Tudo dentro do limite das 20 caixas e dos 25km.

In [74]:
# ================= BLOCO 5: ROTEAMENTO OSRM (RUAS REAIS COM ÍCONES) =================
!pip install polyline

import requests
import folium
import polyline

print("Consultando malha viária na API OSRM. Aguarde...\n")

# 1. Criação de um novo mapa base limpo (Google Maps)
mapa_real = folium.Map(
    location=[-23.2100, -45.8900],
    zoom_start=13,
    tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
    attr='Google Maps'
)
cores_rotas = ['red', 'blue', 'green']

# 2. Adicionando marcadores com ícones personalizados
for i in range(len(dados_locais_reais['Local'])):
    nome_local = dados_locais_reais['Local'][i]

    # Hospital (Pino Preto, Cruz)
    if i == 0:
        icone_cor = 'black'
        icone_tipo = 'plus'
    # Unidades de Saúde (Pino Azul Claro, Estrela)
    elif 'Unidade' in nome_local:
        icone_cor = 'cadetblue'
        icone_tipo = 'star'
    # Residências (Pino Laranja, Casa)
    else:
        icone_cor = 'orange'
        icone_tipo = 'home'

    folium.Marker(
        [dados_locais_reais['Lat'][i], dados_locais_reais['Lon'][i]],
        popup=f"<b>{nome_local}</b><br>Demanda: {dados_locais_reais['Demanda_Caixas'][i]} caixas<br>Prioridade: {dados_locais_reais['Prioridade'][i]}",
        icon=folium.Icon(color=icone_cor, icon=icone_tipo, prefix='glyphicon')
    ).add_to(mapa_real)

# 3. Função para buscar a geometria da rota (traçado da rua) na API OSRM
def obter_rota_osrm(coords):
    # OSRM usa o formato 'longitude,latitude' na URL
    coordenadas_str = ";".join([f"{lon},{lat}" for lat, lon in coords])
    url = f"http://router.project-osrm.org/route/v1/driving/{coordenadas_str}?overview=full"

    try:
        resposta = requests.get(url)
        dados = resposta.json()
        if dados.get("code") == "Ok":
            geometria = polyline.decode(dados["routes"][0]["geometry"])
            return geometria
    except Exception as e:
        print(f"Erro ao buscar rota na OSRM: {e}")
    return None

# 4. Para cada veículo, converte as paradas em caminhos de rua reais
for i, viagem in enumerate(melhores_viagens_vrp):
    rota_coordenadas = [[dados_locais_reais['Lat'][0], dados_locais_reais['Lon'][0]]]
    for loc_idx in viagem:
        rota_coordenadas.append([dados_locais_reais['Lat'][loc_idx], dados_locais_reais['Lon'][loc_idx]])
    rota_coordenadas.append([dados_locais_reais['Lat'][0], dados_locais_reais['Lon'][0]])

    # Busca o contorno das ruas entre os pontos
    caminho_real = obter_rota_osrm(rota_coordenadas)

    # Se a API retornar sucesso, plota o caminho desenhando as ruas
    if caminho_real:
        folium.PolyLine(
            caminho_real,
            color=cores_rotas[i],
            weight=5,
            opacity=0.8,
            tooltip=f"Veículo {i+1} (Roteamento Real)"
        ).add_to(mapa_real)
    else:
        # Fallback de segurança (linha reta se a API falhar)
        folium.PolyLine(rota_coordenadas, color=cores_rotas[i], weight=4, opacity=0.8).add_to(mapa_real)

mapa_real.save('mapa_real_final.html')

print("Roteamento viário concluído com sucesso!")
# Exibe o mapa final
mapa_real

Consultando malha viária na API OSRM. Aguarde...

Roteamento viário concluído com sucesso!


## 📍 5. Bônus de Evolução: Roteamento Real na Malha Viária (OSRM)
Até o bloco 4, nós focamos no processamento matemático do Algoritmo Genético, utilizando a distância euclidiana (linhas retas) para garantir o desempenho computacional durante as 100 gerações de evolução.

No entanto, em um ambiente logístico real, a frota deve respeitar o trânsito. Portanto, neste bloco, evoluímos a solução para nível de mercado.
Integramos a saída do nosso otimizador com a API **OSRM (Open Source Routing Machine)**, que calcula a geometria do trajeto considerando a malha viária real e sentidos de rua. O mapa gerado abaixo converte a lógica pura do algoritmo na realidade operacional das avenidas de São José dos Campos.

In [75]:
from google import genai
from datetime import datetime

# Configuração da API do Google AI Studio
GOOGLE_API_KEY = "AQ.Ab8RN6LozICkM0M94KFjA1lsXAhPcWtwqYCDTpO1Aznsnh9rpg"
client = genai.Client(api_key=GOOGLE_API_KEY)

# Capturando a data de hoje automaticamente
data_hoje = datetime.now().strftime("%d/%m/%Y")

# Conversão da saída estruturada para formato de texto
relatorio_tecnico = ""
for i, viagem in enumerate(melhores_viagens_vrp):
    nomes_locais = [df_locais['Local'][x] for x in viagem]
    carga_total = sum(demandas[x] for x in viagem)
    relatorio_tecnico += f"Veículo {i+1} (Carga Total: {carga_total} caixas): Hospital -> {' -> '.join(nomes_locais)} -> Hospital\n"

# Engenharia de Prompt (System Context) ajustada com a data dinâmica
prompt_logistico = f"""
Atue como o Head de Logística de um hospital universitário.
Recebemos as rotas otimizadas pelo nosso Algoritmo Genético, que processou demandas respeitando capacidade e autonomia.

Dados brutos de saída:
{relatorio_tecnico}

Redija um memorando executivo e claro para os motoristas, contendo:
1. Uma breve introdução motivacional.
2. A escala de cada veículo em bullet points, com horários fictícios coerentes para o turno da manhã de hoje (Data do Memorando: {data_hoje}).
3. Um alerta de segurança sobre a conferência de carga no hospital e o manuseio de insumos médicos frágeis e termolábeis.
4. Mantenha um tom profissional, direto e formal.
"""

print(f"Processando inteligência de linguagem natural via Gemini para a data {data_hoje}...\n")
try:
    response = client.models.generate_content(
        model='gemini-3.8-flash',
        contents=prompt_logistico
    )

    print("================ MEMORANDO OPERACIONAL (GERADO VIA LLM) ================\n")
    print(response.text)
    print("\n=========================================================================")
except Exception as e:
    print(f"Erro de comunicação com a API: {e}")

Processando inteligência de linguagem natural via Gemini para a data 11/10/2026...

Erro de comunicação com a API: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


## 🤖 6. O Toque Humano e Automação (Integração com IA Generativa)
Para finalizar o ecossistema do projeto, resolvemos o último desafio logístico: a comunicação. Motoristas não leem matrizes de código; eles precisam de instruções claras.

Conectamos a nossa solução com a Inteligência Artificial do **Google Gemini (modelo 3.8-flash)**.
Nós passamos os dados geográficos e numéricos das rotas para a IA e, via Engenharia de Prompt, instruímos o modelo a atuar como um *Head de Logística*.

O Gemini traduz os dados brutos e gera automaticamente um memorando operacional em linguagem natural. Ele constrói a escala de horários e redige os alertas de segurança obrigatórios para o manuseio e transporte dos insumos termolábeis da nossa frota.